In [2]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Add_missing_terminal_cappings)=
# Add missing terminal cappings

*Adding terminal cappings to peptides and proteins.*

Capping the termini of peptides and proteins is often required in molecular modeling to ensure charge neutrality, stability, or to match specific biological conditions. The function {func}`molsysmt.build.add_missing_terminal_cappings` adds chemical capping groups (such as `ACE` for N-terminus and `NME` for C-terminus) or completes uncapped terminal residues.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.build.add_missing_terminal_cappings`.
:::


**Choosing a source population.**

`add_missing_terminal_cappings` interprets numeric selections as source **atom
indices**; a Boolean mask spans the source atom axis. Resolve a query with
`msm.select(molsys, selection=...)` to obtain the equivalent indices. The input
syntax changes query parsing, not the meaning of an integer list. Empty
selections return independent unchanged copies.

Native local heavy-atom/OXT/N-bound-H completion enables whole groups touched by
the source atoms. ACE/NME requests enable complete peptide/protein covalent
components touched by those atoms. Atom insertion can reorder atoms, so the
composition transports group membership and re-evaluates component populations;
it does not reuse stale atom indices or equate components with stable molecules.
An internal selected group remains internal. Existing state, placement,
attribute-loss and interaction-invalidation limits still apply.

`get_missing_terminal_cappings` is a **group** assessment: its integer selections
are group indices. It identifies termini from complete chains and then filters
the report; a selection edge is not a chemical terminus.

**Recognize chemical ends from stored bonds.** The experimental {func}`molsysmt.chemical_states.get_peptide_termini` follows named N-CA-C motifs and inter-group C-N links, independently of atom/group storage order or group ID labels. Complete chain context precedes endpoint filtering. Connected cycles have no free ends; disconnected, branched or unsupported backbones are refused. Missing-OXT assessment and terminal addition reuse this rule, while placement still does not assign complete chemistry. See {ref}`Getting peptide termini <Tutorial_Get_peptide_termini>`.


## Basic usage

Let's show how this function works using a tripeptide example (`AlaValPro`):


In [3]:
import molsysmt as msm


In [4]:
molsys = msm.build.build_peptide('AlaValPro', to_form='molsysmt.MolSys')


In [5]:
msm.info(molsys, element='group')


index,id,name,type,n atoms,component index,chain index,molecule index,molecule type,entity index,entity name
0,0,ALA,amino acid,10,0,0,0,peptide,0,peptide
1,1,VAL,amino acid,16,0,0,0,peptide,0,peptide
2,2,PRO,amino acid,14,0,0,0,peptide,0,peptide


As shown, the termini of the peptide are currently incomplete. We now add the atoms required to complete the terminal amino acids without capping groups, producing a charged version of the peptide:


In [6]:
molsys_charged = msm.build.add_missing_terminal_cappings(molsys, N_terminal=None, C_terminal=None)


In [7]:
msm.info(molsys_charged, element='group')


index,id,name,type,n atoms,component index,chain index,molecule index,molecule type,entity index,entity name
0,0,ALA,amino acid,12,0,0,0,peptide,0,peptide
1,1,VAL,amino acid,16,0,0,0,peptide,0,peptide
2,2,PRO,amino acid,15,0,0,0,peptide,0,peptide


Let's check the group charges under these conditions using {func}`molsysmt.physchem.get_charge`:


In [8]:
msm.physchem.get_charge(molsys_charged, element='group')


Magnitude,[0.0 0.0 0.0]
Units,elementary_charge


To neutralize terminal charges, we can cap the N-terminus with an acetyl group (`ACE`) and the C-terminus with an N-methyl amide group (`NME`):


In [9]:
molsys_uncharged = msm.build.add_missing_terminal_cappings(molsys, N_terminal='ACE', C_terminal='NME')


In [10]:
msm.info(molsys_uncharged, element='group')


index,id,name,type,n atoms,component index,chain index,molecule index,molecule type,entity index,entity name
0,0,ACE,terminal capping,6,0,0,0,peptide,0,peptide
1,0,ALA,amino acid,10,0,0,0,peptide,0,peptide
2,1,VAL,amino acid,16,0,0,0,peptide,0,peptide
3,2,PRO,amino acid,14,0,0,0,peptide,0,peptide
4,4,NME,terminal capping,6,0,0,0,peptide,0,peptide


We inspect the group charges again to confirm that the capped peptide is now neutral:


In [11]:
msm.physchem.get_charge(molsys_uncharged, element='group')


Magnitude,[0.0 0.0 0.0 0.0 0.0]
Units,elementary_charge


## Native domain preservation

Native insertion of terminal atoms or ACE/NME requires one chemical state.
Existing atom IDs, chemical assignments, state identity/provenance and coordinates
survive. Inserted groups can change atom/group indices. New chemical values remain
unknown and connectivity is partial: a modeled capping geometry does not declare
or validate all terminal chemistry. Assign the intended chemical state separately.

Named interaction analyses retain their definitions and remapped source axes;
occurrences and evaluated coverage are cleared for recalculation. Structural
attributes or force-field atom parameters without values for new atoms are dropped
with a diagnostic. Observe these diagnostics in preparation workflows. The strict
attribute-loss option belongs to the separate heavy-atom repair tool; this terminal
tool keeps its existing intersection behavior.

### Inspecting nonterminal gaps

The native engine also calls bounded heavy-atom repair. Unsupported side-chain
gaps remain missing with a warning even when OXT or ACE/NME insertion succeeds.
Inspect both `msm.build.get_missing_terminal_cappings(molsys_charged)` and
`msm.build.get_missing_heavy_atoms(molsys_charged)`. PDBFixer can choose wider
heavy-atom reconstruction, so whole-system counts need not match between engines.
Matching terminal inventories do not certify all side-chain coordinates.


:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Build peptide <Tutorial_Build_peptide>`: Generate capped peptide structures from sequence with {func}`molsysmt.build.build_peptide`.
- {ref}`Info <Tutorial_Info>`: Display structural and topological summaries of a molecular system with {func}`molsysmt.basic.info`.
- {ref}`Add missing terminal cappings <Tutorial_Add_missing_terminal_cappings>`: Add terminal cappings to peptides or proteins with {func}`molsysmt.build.add_missing_terminal_cappings`.
- {ref}`Get charge <Tutorial_Get_charge>`: Retrieve formal or net electric charges with {func}`molsysmt.physchem.get_charge`.
:::
